# Step 04 Fixture Overview

Renders all 5 fixtures from the Step 04 matrix (A1, A2, B1, R1, R2) via `proto3.viz.render` and shows them inline for visual comparison.

## Prerequisites

1. `pip install -e ".[dev]"` (one-time, from repo root)
2. `nbstripout --install` (one-time, activates the `*.ipynb filter=nbstripout` registered in `.gitattributes`)
3. The next code cell finds the repo root automatically by walking up; works whether jupyter is launched from repo root or VSCode (cwd = notebook dir).

## Output convention (S03-D13)

Generated SVGs go to `outputs/notebooks/step04_fixture_overview/<run_id>/`. Re-runs create new folders. `outputs/` is gitignored (D014).

In [ ]:
from pathlib import Path


def _find_repo_root(start: Path) -> Path:
    """Walk up from `start` until a directory containing pyproject.toml is found."""
    p = start.resolve()
    for candidate in [p, *p.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError(f"pyproject.toml not found from {start} upward")


ROOT = _find_repo_root(Path.cwd())
print("repo root:", ROOT)

In [ ]:
import sys

if str(ROOT / "tests") not in sys.path:
    sys.path.insert(0, str(ROOT / "tests"))
from fixture_matrix import MATRIX, fixture_path

for mid, meta in MATRIX.items():
    print(f"{mid:>3}  {meta['file']:<32}  expected_failure={meta['expected_failure']}")

In [ ]:
from datetime import datetime

from proto3.schema.input import BuildingInput
from proto3.schema.serialize import from_json
from proto3.viz import render

run_id = datetime.now().strftime("%Y%m%dT%H%M%S")
out_dir = ROOT / "outputs" / "notebooks" / "step04_fixture_overview" / run_id
out_dir.mkdir(parents=True, exist_ok=True)

svg_paths = {}
for mid in MATRIX:
    p = fixture_path(mid)
    building = from_json(BuildingInput, p)
    out_path = out_dir / f"{mid}_{p.stem}.svg"
    render(building, out_path=str(out_path))
    svg_paths[mid] = out_path

for mid, out_path in svg_paths.items():
    print(f"{mid}  {out_path.relative_to(ROOT)}")

In [ ]:
from IPython.display import SVG, Markdown, display

for mid, out_path in svg_paths.items():
    meta = MATRIX[mid]
    display(Markdown(f"### {mid} \u2014 `{meta['file']}` (expected_failure={meta['expected_failure']})"))
    display(SVG(filename=str(out_path)))

---

## Notes

- All 5 fixtures share the 12-layer SVG order from D013. Step 04 only populates layer 0 (footprint) + layer 11 (atoms grid); other layers stay empty until later Stages land.
- R1 (`apartment_no_bath.json`) and R2 (`apartment_too_small.json`) render fine — the SVG renderer does not run cardinality or area gates. Their failure detection lives in Stage 01 (R1, Step 04) and Stage 02 (R2, Step 06). See `tests/fixture_matrix.py` for `expected_failure` metadata.
- Re-running creates a new `run_id` folder under `outputs/notebooks/step04_fixture_overview/`. Old runs are kept.